# Fraud Detection: What the Final Model Looks At

Picture for the stakeholder presentation. It answers: **which clients' properties does the final model use most, and what do they mean for the fraud risk?**

- **Model:** the final LightGBM from [14_final-model-submission.ipynb](14_final-model-submission.ipynb), loaded from `models/lightgbm_final.txt` (nothing is trained again).
- **Importance** = *gain*: how much the model's predictions improved through all the decisions that use this feature. We show the share of the total (%).
- **Fraud rate** = share of clients with fraud in the group, on all 135,493 labelled clients. These are links, not proof of cause.

The features are rebuilt from the cleaned tables with the same functions as in notebooks 06, 11 and 13 (`features.py`).

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
import matplotlib.pyplot as plt

from features import (build_diversity_features, build_extra_features, build_features,
                      categories_to_keep, group_rare)

CLEAN_DIR = Path("data/clean")
MODEL_DIR = Path("models")
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(exist_ok=True)

## 1. Rebuild the 39 features for all labelled clients

Same steps as in the earlier notebooks. The rare tariff plans are grouped as "other" (plans below 1% of clients), as in notebook 06. The columns are put in the order the model was trained with.

In [ ]:
client = pd.read_csv(CLEAN_DIR / "client_train_clean.csv", parse_dates=["creation_date"])
invoice = pd.read_csv(CLEAN_DIR / "invoice_train_clean.csv", parse_dates=["invoice_date"], low_memory=False)

# one row per client: base features, time and meter features, diversity features
X = build_features(client, invoice).join(build_extra_features(client, invoice)).join(build_diversity_features(client, invoice))
X["tarif_type"] = group_rare(X["tarif_type"], categories_to_keep(X["tarif_type"]))

features = json.loads((MODEL_DIR / "selected_features_v3.json").read_text())["features"]
X = X[features]
y = client.set_index("client_id")["target"].loc[X.index]

assert X.shape == (135_493, 39) and X.index.is_unique
print(f"{X.shape[0]:,} clients, {X.shape[1]} features, fraud rate {y.mean():.1%}")

## 2. Importance of the 39 features in the final model

The saved model has the same 39 feature names. We check that before we trust the ranking.

In [ ]:
booster = lgb.Booster(model_file=str(MODEL_DIR / "lightgbm_final.txt"))
assert booster.feature_name() == features, "the saved model must use the same 39 features"

gain = pd.Series(booster.feature_importance("gain"), index=features)
importance = (100 * gain / gain.sum()).sort_values(ascending=False)
importance.head(12).round(1).rename("gain share (%)").to_frame()

## 3. The picture

**Left:** the 10 features the model relies on most, in plain words and coloured by topic.
**Right:** for the four most telling ones, the fraud rate per group. The dashed line is the average of all clients (5.6%).

Colours: one colour per topic, chosen to be told apart by people with colour blindness. The bar labels carry the same information, so colour is never the only signal.

In [ ]:
# plain-language names for the features that can appear in the top 10
LABELS = {
    "n_counters": "Number of meters (replaced meter)",
    "days_since_last_bill": "Days since the last bill",
    "n_codes": "Different meter codes used",
    "region": "Region",
    "consommation_level_1_sum": "Total consumption, level 1",
    "consommation_level_3_sum": "Total consumption, level 3",
    "client_age_years": "Age of the client account",
    "cons_std_gaz": "Gas: ups and downs in use",
    "max_change_gaz": "Gas: biggest jump between bills",
    "gap_max_days": "Longest break between bills",
    "history_years": "Years of billing history",
    "statue_unusual_share": "Unusual meter status",
    "client_catg": "Client category",
}
# topic of every feature (all 39 are covered, checked below)
TOPICS = {
    "Meters and meter setup": ["n_counters", "n_codes", "n_remarque", "n_tarif", "remarque_8_share", "statue_unusual_share"],
    "Billing activity and history": ["n_bills", "days_since_last_bill", "share_last_2y", "bills_in_2016_17", "inv_per_date",
                                      "history_years", "bills_per_year", "gap_mean_days", "gap_max_days",
                                      "months_number_mean", "months_number_max"],
    "Consumption": ["cons_mean", "cons_median", "cons_std", "cons_min", "cons_max", "consommation_level_1_sum",
                    "consommation_level_2_sum", "consommation_level_3_sum", "consommation_level_4_sum",
                    "max_change", "cons_std_elec", "max_change_elec", "cons_std_gaz", "max_change_gaz",
                    "cons_cv", "level2_3_share", "level4_share", "cons_p90_over_median"],
    "Client profile": ["region", "client_catg", "client_age_years", "tarif_type"],
}
assert sorted(c for cols in TOPICS.values() for c in cols) == sorted(features)
topic_of = {c: topic for topic, cols in TOPICS.items() for c in cols}
# colour-blind-safe colours, one per topic
COLORS = {"Meters and meter setup": "#0072B2", "Billing activity and history": "#E69F00",
          "Consumption": "#009E73", "Client profile": "#7F7F7F"}
INK, MUTED = "#222222", "#666666"

top10 = importance.head(10)
print([f for f in top10.index if f not in LABELS])

In [ ]:
overall = y.mean() * 100
frame = X.assign(fraud=y)

def rate_table(groups):
    # fraud rate (%) and number of clients per group
    table = frame.groupby(groups, observed=True)["fraud"].agg(["mean", "size"])
    return (table["mean"] * 100).values, table["size"].values

# four feature stories: (feature, plain title, group labels, group assignment)
stories = [
    ("n_counters", "Number of meters per client",
     ["1", "2", "3", "4 or more"], frame["n_counters"].clip(upper=4)),
    ("n_codes", "Different meter codes used",
     ["1", "2", "3 or more"], frame["n_codes"].clip(upper=3)),
    ("days_since_last_bill", "Time since the last bill",
     ["under 5 years", "over 5 years"], (frame["days_since_last_bill"] > 5 * 365).astype(int)),
    ("consommation_level_1_sum", "Total consumption, level 1",
     ["lowest 25%", "low", "high", "highest 25%"],
     pd.qcut(frame["consommation_level_1_sum"], 4, labels=False)),
]

fig = plt.figure(figsize=(15, 7.5), facecolor="white")
grid = fig.add_gridspec(2, 3, width_ratios=[1.25, 0.8, 0.8], hspace=0.5, wspace=0.35)

# left: top 10 features by gain share
ax = fig.add_subplot(grid[:, 0])
names = [LABELS[f] for f in top10.index][::-1]
values = top10.values[::-1]
colors = [COLORS[topic_of[f]] for f in top10.index][::-1]
ax.barh(names, values, color=colors, height=0.62)
for i, v in enumerate(values):
    ax.text(v + 0.25, i, f"{v:.1f}%", va="center", fontsize=10, color=INK)
ax.set_title("What the model looks at most", loc="left", fontsize=14, fontweight="bold", color=INK)
ax.set_xlabel("Share of the model's decisions that rely on this feature (%)", color=MUTED)
ax.set_xlim(0, values.max() * 1.15)
ax.tick_params(axis="y", length=0, labelsize=10.5, labelcolor=INK)
ax.tick_params(axis="x", colors=MUTED)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#cccccc")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORS.values()]
ax.legend(handles, COLORS.keys(), loc="lower right", frameon=False, fontsize=9.5, labelcolor=INK)

# right: fraud rate per group for four features
for k, (feature, title, group_labels, assignment) in enumerate(stories):
    ax = fig.add_subplot(grid[k // 2, 1 + k % 2])
    rates, sizes = rate_table(assignment)
    assert len(rates) == len(group_labels), feature
    color = COLORS[topic_of[feature]]
    ax.bar(group_labels, rates, color=color, width=0.62)
    ax.axhline(overall, color=MUTED, linestyle="--", linewidth=1)
    for i, r in enumerate(rates):
        ax.text(i, r + 0.6, f"{r:.1f}%", ha="center", fontsize=9.5, color=INK,
                bbox=dict(facecolor="white", edgecolor="none", pad=1.2))
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold", color=INK)
    ax.set_ylim(0, 23)  # the same scale in all four panels, so the bars can be compared
    ax.set_yticks([])
    ax.tick_params(axis="x", labelsize=9, colors=INK, length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color("#cccccc")
    if k == 1:
        ax.text(-0.45, overall + 0.6, "average 5.6%", ha="left", fontsize=8.5, color=MUTED)

fig.text(0.552, 0.915, "Share of clients with fraud, per group", fontsize=14, fontweight="bold", color=INK)
fig.text(0.02, 0.985, "Meters, billing activity and consumption tell the model who to inspect first",
         ha="left", fontsize=16, fontweight="bold", color=INK)
fig.savefig(FIGURE_DIR / "feature_story.png", dpi=170, bbox_inches="tight", facecolor="white")
plt.show()

**Reading the picture**
- Clients with **3 or more meters** are fraud 7 to 10 times more often than clients with one meter. A replaced meter can be a sign of tampering, **or** of STEG's reaction after it found fraud (see the check in [11_feature-selection.ipynb](11_feature-selection.ipynb)).
- Clients with **3 or more different meter codes** are fraud far more often than clients with one code (diversity features, [13_diversity-features.ipynb](13_diversity-features.ipynb)).
- Clients whose **last bill is more than 5 years old** are almost never fraud. Closed accounts are probably rarely inspected, so they are rarely *labelled* fraud.
- **Higher consumption** goes with more fraud, but the other three signals are stronger.

## 4. Backup: plain correlation with fraud

A correlation is one number per feature between -1 and 1 (0 = no straight-line link). We use the **rank correlation** (Spearman), because many features are very skewed. Single features are only weakly linked to fraud. The model reaches ROC-AUC 0.876 because it **combines** many weak signals, so the importance picture above is the fairer one for the presentation. This table is for questions from the audience.

In [ ]:
spearman = X[top10.index].corrwith(y, method="spearman")
backup = pd.DataFrame({"feature": [LABELS[f] for f in top10.index],
                       "importance (%)": top10.values.round(1),
                       "rank correlation with fraud": spearman.values.round(2)})
backup